# Avocado Prices — Analyze Phase

Business question: **how do avocado prices and volume vary by region, season, and type — and
what does the price-volume relationship suggest about demand?** Uses the cleaned dataset from the
Process phase, filtering by `region_tier` at each step to avoid double-counting.

## Setup

In [1]:
import pandas as pd
import os

PROC = "../data/processed"
SUMMARY_DIR = "../data/summary"
os.makedirs(SUMMARY_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(PROC, "avocado_clean.parquet"))
national = df[df["region_tier"] == "national"].copy()
national["month"] = national["date"].dt.month
print(f"Loaded: {df.shape}, national subset: {national.shape}")

Loaded: (18249, 14), national subset: (338, 15)

## 1. Average price by type

In [2]:
price_by_type = national.groupby("type")["average_price"].agg(["mean", "median", "std"]).round(3)
price_by_type.to_csv(os.path.join(SUMMARY_DIR, "price_by_type.csv"))
price_by_type

               mean  median    std
type
conventional  1.092    1.04  0.174
organic       1.546    1.53  0.203

**Organic averages $1.55/unit vs. $1.09 for conventional — a ~42% premium.**

## 2. Seasonality — average price by month

In [3]:
seasonal = national.groupby(["month", "type"])["average_price"].mean().unstack().round(3)
seasonal.to_csv(os.path.join(SUMMARY_DIR, "seasonal_price.csv"))
seasonal

type   conventional  organic
month
1             0.994    1.466
2             0.936    1.444
3             1.054    1.425
4             1.062    1.474
5             1.025    1.484
6             1.078    1.597
7             1.166    1.472
8             1.180    1.609
9             1.242    1.796
10            1.301    1.742
11            1.142    1.644
12            1.003    1.511

Both types peak in **September/October** and bottom out in **January/February** — a clear seasonal pattern.

## 3. Price trend over time

In [4]:
national["year_month"] = national["date"].dt.to_period("M").astype(str)
trend = national.groupby(["year_month", "type"])["average_price"].mean().unstack().round(3)
trend.to_csv(os.path.join(SUMMARY_DIR, "price_trend_monthly.csv"))
trend.tail(12)

type        conventional  organic
year_month
2017-04            1.202    1.526
2017-05            1.205    1.630
2017-06            1.200    1.670
2017-07            1.268    1.754
2017-08            1.385    1.940
2017-09            1.580    2.007
2017-10            1.520    1.872
2017-11            1.188    1.780
2017-12            1.070    1.578
2018-01            1.125    1.585
2018-02            0.995    1.545
2018-03            1.060    1.532

Conventional prices climbed sharply through 2017, peaking at $1.58 in September 2017 — well above the 2015-2016 range (typically $0.87-$1.15). This matches the well-documented 2017 avocado supply shortage. Note: national organic price is exactly $1.00 for all 4 weeks of July 2015 — an unusually round, unvarying figure versus every other month, likely a placeholder/reporting gap rather than a genuine price; visible in the chart but not a real signal.

## 4. Regional price variation

In [5]:
major = df[df["region_tier"] == "major_region"]
regional_price = major.groupby(["region", "type"])["average_price"].mean().unstack().round(3)
regional_price["gap"] = (regional_price["organic"] - regional_price["conventional"]).round(3)
regional_price = regional_price.sort_values("conventional", ascending=False)
regional_price.to_csv(os.path.join(SUMMARY_DIR, "regional_price.csv"))
regional_price

type          conventional  organic    gap
region
Northeast            1.344    1.859  0.515
Midsouth             1.207    1.602  0.395
GreatLakes            1.182    1.495  0.313
Plains                1.166    1.708  0.542
Southeast              1.163    1.633  0.470
California            1.105    1.685  0.580
West                    0.985    1.559  0.574
SouthCentral            0.869    1.333  0.464

**Northeast pays 55% more than SouthCentral for conventional avocados** ($1.34 vs $0.87) — the widest regional spread in the data. California, the main growing region, is mid-pack, not cheapest.

## 5. Price vs. volume relationship

In [6]:
nat_conv = national[national["type"] == "conventional"]
corr = nat_conv["average_price"].corr(nat_conv["total_volume"])
print(f"Correlation between average_price and total_volume (national, conventional): {corr:.3f}")

Correlation between average_price and total_volume (national, conventional): -0.510

A moderate negative relationship, consistent with a normal demand curve: weeks with higher prices tend to see lower volume sold, and vice versa.

## 6. Volume trend and organic's share

In [7]:
vol_trend = national.groupby(["year", "type"])["total_volume"].sum().unstack()
vol_trend_millions = (vol_trend / 1e6).round(2)
vol_trend_millions.to_csv(os.path.join(SUMMARY_DIR, "volume_by_year.csv"))
print(vol_trend_millions)
print("\nNote: 2015 and 2018 are partial years (2018 data ends March 25).")

vol_share = national.groupby("type")["total_volume"].sum()
vol_share_pct = (vol_share / vol_share.sum() * 100).round(2)
vol_share_pct.to_csv(os.path.join(SUMMARY_DIR, "volume_share_by_type.csv"))
print()
print(vol_share_pct)

type  conventional  organic
year
2015       1623.69    33.57
2016       1770.26    48.90
2017       1801.77    62.92
2018        505.51    18.13

Note: 2015 and 2018 are partial years (2018 data ends March 25).

type
conventional    97.21
organic          2.79
Name: total_volume, dtype: float64

Organic volume grew from 33.6M to 62.9M units/year (2015→2017, +87%) — nearly double the
conventional growth rate (1,624M→1,802M, +11%) — but organic still makes up only 2.8% of total
volume even after that growth.

## Summary of key findings

1. **Organic carries a ~42% price premium** over conventional ($1.55 vs $1.09 average).
2. **Clear seasonality**: both types peak in September/October, bottom out in January/February.
3. **2017 saw a sharp price spike** (conventional peaked at $1.58 in September 2017, well above
   the 2015-2016 range) — consistent with the documented 2017 supply shortage.
4. **Regional prices vary widely**: Northeast pays 55% more than SouthCentral for conventional
   avocados; California (the main growing region) is mid-pack, not the cheapest.
5. **Price and volume move inversely** (r=-0.51) — a normal demand-curve signal.
6. **Organic is a small but fast-growing segment**: only 2.8% of volume, but growing ~87% over
   2015-2017 versus ~11% for conventional.